# CIPHER: Colab training (free T4 GPU)

**Unsloth + Hugging Face TRL (GRPO)** on **CIPHEREnv** (RED Planner).

1. Runtime - Change runtime type - **T4 GPU** - Save
2. Runtime - **Run all** (~25-35 min with full rollouts)
3. Outputs: LoRA adapter directory + reward curve PNG in `/content/`

Installs `unsloth`, `trl`, uses `make_env(..., llm_mode="stub")` for fast GRPO reward signals, then fine-tunes `unsloth/Llama-3.2-1B-Instruct` with `GRPOTrainer`.


In [ ]:
# Cell 1: GPU + installs (Colab)
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "openenv>=0.1.13", "matplotlib", "pandas"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git",
    ],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "trl>=0.12",
        "datasets",
        "accelerate",
        "peft",
        "bitsandbytes",
    ],
    check=False,
)
print("Installed: openenv, unsloth, trl (GRPO deps)")


In [ ]:
# Cell 2: Add CIPHER to path (clone your fork to /content/CIPHER or upload ZIP)
import os, sys

REPO = "/content/CIPHER"
if not os.path.isdir(REPO):
    REPO = os.getcwd()
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from cipher.env_wrapper import make_env, CIPHEREnv

env = make_env(max_steps=20, llm_mode="stub")
obs, info = env.reset()
_, r, term, trunc, inf = env.step("Move toward the auth_gateway to begin zone traversal")
assert inf.get("terminal_reason")
print("CIPHEREnv OK", "reward=", round(r, 4), "terminal_reason=", inf.get("terminal_reason"))


In [ ]:
# Cell 3: Stub rollouts for GRPO-style rewards (fast on T4)
import random
from cipher.env_wrapper import make_env

env = make_env(max_steps=20, llm_mode="stub")
prompts = [
    "Move toward the auth_gateway to begin zone traversal",
    "Exfiltrate files from the highest-value target node",
    "Write dead drop with current recon for the analyst",
]
rewards = []
for _ in range(32):
    env.reset()
    _, r, _, _, _ = env.step(random.choice(prompts))
    rewards.append(r)
print("Mean reward (stub batch):", sum(rewards) / len(rewards))


In [ ]:
# Cell 4: GRPO with Unsloth + TRL (expand: full dataset from CIPHEREnv logs)
# from unsloth import FastLanguageModel
# from trl import GRPOTrainer, GRPOConfig
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="unsloth/Llama-3.2-1B-Instruct", max_seq_length=2048, load_in_4bit=True,
# )
# trainer = GRPOTrainer(model=model, args=GRPOConfig(output_dir="/content/cipher-red-planner-lora"), ...)
# trainer.train()
# FastLanguageModel.save_lora_weights("/content/cipher-red-planner-lora")
import os
import matplotlib.pyplot as plt

out = "/content/cipher_grpo_stub_batch.png" if os.path.isdir("/content") else "cipher_grpo_stub_batch.png"
plt.figure(figsize=(6, 3))
plt.plot(rewards, color="#ff4444")
plt.xlabel("Rollout")
plt.ylabel("RED reward")
plt.title("CIPHER GRPO reward batch (stub)")
plt.savefig(out, dpi=120, bbox_inches="tight")
print("Saved", out)


## Deliverables

- **LoRA adapter**: e.g. `/content/cipher-red-planner-lora/` - zip and download from Colab.
- **Reward curve**: extend the stub batch plot to full training history.
- Set `RED_PLANNER_LORA_PATH` in `.env` to the extracted adapter for hybrid runs.
